In [0]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DecimalType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
spark = (
    SparkSession.builder
    .appName("CarSales_StarSchema")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "8")  # tune for your cluster size
print("✅ Spark session ready — version:", spark.version)


In [0]:
# # ── Update these paths to match your ADLS / DBFS mount
# BASE_PATH        = "/Volumes/workspace/fact_to_dim&fact"            # DBFS  (Databricks)
# # BASE_PATH      = "abfss://container@account.dfs.core.windows.net/carsales"  # ADLS Gen2

# RAW_PATH         = f"{BASE_PATH}/source_files"
# DIM_PATH         = f"{BASE_PATH}/gold/dimensions"
# FACT_PATH        = f"{BASE_PATH}/gold/facts"

# SALES_FILE       = f"{RAW_PATH}/SalesData_cars.csv"
# INCREMENTAL_FILE = f"{RAW_PATH}/IncrementalSales_cars.csv"

# # Delta table paths
# DIM_BRANCH_PATH  = f"{DIM_PATH}/dim_branch"
# DIM_DEALER_PATH  = f"{DIM_PATH}/dim_dealer"
# DIM_PRODUCT_PATH = f"{DIM_PATH}/dim_product"
# DIM_DATE_PATH    = f"{DIM_PATH}/dim_date"
# FACT_SALES_PATH  = f"{FACT_PATH}/fact_sales"

# print("✅ Paths configured")

In [0]:
raw_schema = StructType([
    StructField("Branch_ID",    StringType(),  True),
    StructField("Dealer_ID",    StringType(),  True),
    StructField("Model_ID",     StringType(),  True),
    StructField("Revenue",      LongType(),    True),
    StructField("Units_Sold",   IntegerType(), True),
    StructField("Date_ID",      StringType(),  True),
    StructField("Day",          IntegerType(), True),
    StructField("Month",        IntegerType(), True),
    StructField("Year",         IntegerType(), True),
    StructField("BranchName",   StringType(),  True),
    StructField("DealerName",   StringType(),  True),
    StructField("Product_Name", StringType(),  True),
])

print("✅ Schema defined — columns:", len(raw_schema.fields))

In [0]:
# import os

# # Check what exists in the volumes path
# volumes_base = "/Volumes/workspace/fact_to_dim&fact"
# try:
#     if os.path.exists(volumes_base):
#         print(f"✅ Base path exists: {volumes_base}")
#         for item in os.listdir(volumes_base):
#             full_path = os.path.join(volumes_base, item)
#             if os.path.isdir(full_path):
#                 print(f"  📁 {item}/")
#                 # List contents of subdirectories
#                 try:
#                     for sub_item in os.listdir(full_path):
#                         print(f"    - {sub_item}")
#                 except:
#                     pass
#             else:
#                 print(f"  📄 {item}")
#     else:
#         print(f"❌ Path does not exist: {volumes_base}")
# except Exception as e:
#     print(f"❌ Error accessing path: {e}")

In [0]:
# # Dynamically load either initial or incremental file
# def load_sales_data(use_incremental=False):
#     file_path = INCREMENTAL_FILE if use_incremental else SALES_FILE
#     df = (
#         spark.read
#         .option("header", "true")
#         .option("encoding", "UTF-8")
#         .schema(raw_schema)
#         .csv(file_path)
#     )
#     print(f"✅ Loaded data from {file_path} — rows: {df.count():,}")
#     df.printSchema()
#     display(df)
#     return df

# # Set which file to load:
# # Set use_incremental=True to load incremental file, False for initial file
# use_incremental = False
# df_raw = load_sales_data(use_incremental)

In [0]:
# # Dynamically load either initial or incremental file
# def load_sales_data():
#     # Check if fact_sales Delta table exists and has data
#     table_name = "workspace.`fact_to_dim&fact`.fact_sales"
#     try:
#         df_existing = spark.table(table_name)
#         has_data = df_existing.count() > 0
#     except Exception:
#         has_data = False

#     file_path = INCREMENTAL_FILE if has_data else SALES_FILE
#     df = (
#         spark.read
#         .option("header", "true")
#         .option("encoding", "UTF-8")
#         .schema(raw_schema)
#         .csv(file_path)
#     )
#     print(f"✅ Loaded data from {file_path} — rows: {df.count():,}")
#     df.printSchema()
#     display(df)
#     return df

# df_raw = load_sales_data()

In [0]:
spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/fact_to_dim&fact/source_files/schema/") \
    .schema(raw_schema) \
    .load("/Volumes/workspace/fact_to_dim&fact/source_files") \
    .writeStream \
    .option("checkpointLocation", "/Volumes/workspace/fact_to_dim&fact/source_files/checkpoint/") \
    .option("mergeSchema", "true") \
    .trigger(once=True) \
    .toTable("workspace.`fact_to_dim&fact`.fact_sales_staging") \
    .awaitTermination()

In [0]:
df_raw_batch = spark.table("workspace.`fact_to_dim&fact`.fact_sales_staging")

In [0]:

# ── 5a: Null checks on critical columns
print("─── NULL CHECK ───────────────────────────────")
critical_cols = ["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID", "Revenue"]
for col in critical_cols:
    null_count = df_raw_batch.filter(F.col(col).isNull()).count()
    print(f"  {col:<15} → {null_count} nulls")

In [0]:
# ── 5b: Drop rows where ANY key column is null
df_clean = df_raw_batch.dropna(subset=["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID"])


In [0]:
# ── 5c: Trim whitespace from string columns
string_cols = ["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID",
               "BranchName", "DealerName", "Product_Name"]
for col in string_cols:
    df_clean = df_clean.withColumn(col, F.trim(F.col(col)))


In [0]:
# Recreate df_clean from batch data
df_clean_batch = df_raw_batch.dropna(subset=["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID"])

string_cols = ["Branch_ID", "Dealer_ID", "Model_ID", "Date_ID",
               "BranchName", "DealerName", "Product_Name"]
for col in string_cols:
    df_clean_batch = df_clean_batch.withColumn(col, F.trim(F.col(col)))

# ── Remove exact duplicate rows
df_clean_batch = df_clean_batch.dropDuplicates()

print(f"\n✅ After cleaning — rows: {df_clean_batch.count():,}")

In [0]:
## 🏢 Cell 6 — Create DimBranch

df_dim_branch = (
    df_clean_batch
    .select("Branch_ID", "BranchName")
    .dropDuplicates(["Branch_ID"])           # deduplicate on natural key
    .withColumn("DimBranch_SK",              # surrogate key
                F.monotonically_increasing_id()+1)
    .withColumn("Created_At", F.current_timestamp())
    .select("DimBranch_SK", "Branch_ID", "BranchName", "Created_At")
    .orderBy("Branch_ID")
)

print(f"✅ DimBranch rows: {df_dim_branch.count():,}")
df_dim_branch.show(5, truncate=False)

In [0]:
# ── Write as Delta table (creates if not exists)
(
    df_dim_branch.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_branch")
)
print("✅ DimBranch saved → workspace.`fact_to_dim&fact`.dim_branch")

In [0]:
## 🏬 Cell 7 — Create DimDealer

df_dim_dealer = (
    df_clean_batch
    .select("Dealer_ID", "DealerName")
    .dropDuplicates(["Dealer_ID"])
    .withColumn("DimDealer_SK", F.monotonically_increasing_id())
    .withColumn("Created_At", F.current_timestamp())
    .select("DimDealer_SK", "Dealer_ID", "DealerName", "Created_At")
    .orderBy("Dealer_ID")
)

print(f"✅ DimDealer rows: {df_dim_dealer.count():,}")
df_dim_dealer.show(5, truncate=False)

(
    df_dim_dealer.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_dealer")
)
print("✅ DimDealer saved → workspace.`fact_to_dim&fact`.dim_dealer")

In [0]:
## 🚗 Cell 8 — Create DimProduct

df_dim_product = (
    df_clean_batch
    .select("Model_ID", "Product_Name")
    .dropDuplicates(["Model_ID"])
    .withColumn("DimProduct_SK", F.monotonically_increasing_id())
    .withColumn("Created_At", F.current_timestamp())
    .select("DimProduct_SK", "Model_ID", "Product_Name", "Created_At")
    .orderBy("Model_ID")
)

print(f"✅ DimProduct rows: {df_dim_product.count():,}")
df_dim_product.show(5, truncate=False)

(
    df_dim_product.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_product")
)
print("✅ Dimproduct saved → workspace.`fact_to_dim&fact`.dim_product")


In [0]:
## 📅 Cell 9 — Create DimDate  (with full calendar enrichment)

df_dim_date = (
    df_clean_batch
    .select("Date_ID", "Day", "Month", "Year")
    .dropDuplicates(["Date_ID"])
    # ── Build a proper date column
    .withColumn(
        "Full_Date",
        F.to_date(
            F.concat_ws("-", F.col("Year"), F.col("Month"), F.col("Day")),
            "yyyy-M-d"
        )
    )
    # ── Enrich with derived calendar attributes
    .withColumn("Quarter",       F.quarter(F.col("Full_Date")))
    .withColumn("Day_Of_Week",   F.dayofweek(F.col("Full_Date")))   # 1=Sun … 7=Sat
    .withColumn("Day_Name",      F.date_format(F.col("Full_Date"), "EEEE"))
    .withColumn("Month_Name",    F.date_format(F.col("Full_Date"), "MMMM"))
    .withColumn("Is_Weekend",
                F.when(F.col("Day_Of_Week").isin([1, 7]), True).otherwise(False))
    .withColumn("DimDate_SK",    F.monotonically_increasing_id())
    .withColumn("Created_At",    F.current_timestamp())
    .select(
        "DimDate_SK", "Date_ID", "Full_Date",
        "Day", "Month", "Year", "Quarter",
        "Day_Of_Week", "Day_Name", "Month_Name", "Is_Weekend",
        "Created_At"
    )
    .orderBy("Full_Date")
)

print(f"✅ DimDate rows: {df_dim_date.count():,}")
df_dim_date.show(5, truncate=False)

(
    df_dim_date.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.dim_date")
)
print("✅ Dimdate saved → workspace.`fact_to_dim&fact`.dim_date")

In [0]:
# ── Re-read dimension tables from Delta (single source of truth for SKs)
dim_branch_sk  = spark.table("workspace.`fact_to_dim&fact`.dim_branch")\
                      .select("Branch_ID",  "DimBranch_SK")
dim_dealer_sk  = spark.table("workspace.`fact_to_dim&fact`.dim_dealer")\
                      .select("Dealer_ID",  "DimDealer_SK")
dim_product_sk = spark.table("workspace.`fact_to_dim&fact`.dim_product")\
                      .select("Model_ID",   "DimProduct_SK")
dim_date_sk    = spark.table("workspace.`fact_to_dim&fact`.dim_date")\
                      .select("Date_ID",    "DimDate_SK")

In [0]:
# ── Join all dimension SKs onto the clean source
df_fact_sales = (
    df_clean_batch
    .join(dim_branch_sk,  on="Branch_ID",  how="left")
    .join(dim_dealer_sk,  on="Dealer_ID",  how="left")
    .join(dim_product_sk, on="Model_ID",   how="left")
    .join(dim_date_sk,    on="Date_ID",    how="left")
    # ── Surrogate key for the fact row itself
    .withColumn(
        "Sale_SK",
        F.row_number().over(
            Window.orderBy("Date_ID", "Branch_ID", "Dealer_ID")
        )
    )
    .withColumn("Created_At", F.current_timestamp())
    # ── Keep only the fact columns (SKs as FKs + measures)
    .select(
        "Sale_SK",
        "DimBranch_SK",
        "DimDealer_SK",
        "DimProduct_SK",
        "DimDate_SK",
        # Natural keys kept for traceability (optional — remove in strict DW)
        "Branch_ID", "Dealer_ID", "Model_ID", "Date_ID",
        # ── MEASURES
        "Revenue",
        "Units_Sold",
        "Created_At"
    )
)

print(f"✅ FactSales rows: {df_fact_sales.count():,}")
df_fact_sales.show(5, truncate=False)

(
    df_fact_sales.write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.`fact_to_dim&fact`.fact_sales")
)
print("✅ factsales saved → workspace.`fact_to_dim&fact`.fact_sales")


In [0]:
## ✅ Cell 11 — Verification: Row Count & Sample Joins

print("═" * 55)
print("  STAR SCHEMA — TABLE SUMMARY")
print("═" * 55)

tables = {
    "DimBranch":  "workspace.`fact_to_dim&fact`.dim_branch",
    "DimDealer":  "workspace.`fact_to_dim&fact`.dim_dealer",
    "DimProduct": "workspace.`fact_to_dim&fact`.dim_product",
    "DimDate":    "workspace.`fact_to_dim&fact`.dim_date",
    "FactSales":  "workspace.`fact_to_dim&fact`.fact_sales",
}
for name, table in tables.items():
    count = spark.table(table).count()
    print(f"  {name:<15} → {count:>6,} rows")

print("═" * 55)

In [0]:
# Load all tables in the fact_to_dim&fact catalog
dim_branch  = spark.table("workspace.`fact_to_dim&fact`.dim_branch")
dim_dealer  = spark.table("workspace.`fact_to_dim&fact`.dim_dealer")
dim_product = spark.table("workspace.`fact_to_dim&fact`.dim_product")
dim_date    = spark.table("workspace.`fact_to_dim&fact`.dim_date")
fact_sales  = spark.table("workspace.`fact_to_dim&fact`.fact_sales")

# Display each table
display(dim_branch)
display(dim_dealer)
display(dim_product)
display(dim_date)
display(fact_sales)